# Vietnamese Summarization Fine-tuning on Kaggle

Workflow nay clone repo GitHub, cai dung dependency, chay nhieu experiments tren T4x2, va tong hop ROUGE vao CSV/Markdown.

Can bat Kaggle Internet va GPU T4x2. Dataset parquet phai nam o duong dan da khai bao ben duoi.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True  # set False neu muon giu repo local; outputs nam ngoai repo nen khong bi xoa

WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
OUTPUT_ROOT = WORKING / 'summarization_outputs'
DATA_DIR = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization')
TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VALID_FILE = DATA_DIR / 'valid-00000-of-00001.parquet'

os.chdir(WORKING)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    if check and returncode != 0:
        tail = ''.join(lines[-100:])
        raise RuntimeError(f'Command failed with exit code {returncode}: {cmd}\nLast log lines:\n{tail}')
    return returncode

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

def prepare_repo():
    os.chdir(WORKING)
    if REFRESH_REPO and WORKING_REPO.exists():
        shutil.rmtree(WORKING_REPO)
    if not is_repo(WORKING_REPO):
        run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
    else:
        run('git pull --ff-only', cwd=WORKING_REPO, check=False)
    return WORKING_REPO

repo = prepare_repo()
run('git log --oneline -1', cwd=repo)
print('Repo:', repo)
print('Output root:', OUTPUT_ROOT)


CMD: git clone --depth 1 https://github.com/Anhnguyen0812/pretrained-summarization.git /kaggle/working/pretrained-summarization
Cloning into '/kaggle/working/pretrained-summarization'...
CMD: git log --oneline -1
37a2f2c Add experiment result summarizer
Repo: /kaggle/working/pretrained-summarization
Output root: /kaggle/working/summarization_outputs


In [2]:
os.chdir(repo)
print('CWD:', Path.cwd())
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade --force-reinstall --no-deps transformers==4.46.3 tokenizers==0.20.3', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip show transformers tokenizers sentencepiece | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)


CWD: /kaggle/working/pretrained-summarization
CMD: /usr/bin/python3 -m pip install -q --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.1 MB/s eta 0:00:00
CMD: /usr/bin/python3 -m pip install -q --upgrade --force-reinstall --no-deps transformers==4.46.3 tokenizers==0.20.3
CMD: /usr/bin/python3 -m pip install -q -e .
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.

0

In [3]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

run('nvidia-smi', check=False)
print('TRAIN_FILE:', TRAIN_FILE, 'FOUND' if TRAIN_FILE.exists() else 'MISSING')
print('VALID_FILE:', VALID_FILE, 'FOUND' if VALID_FILE.exists() else 'MISSING')
if not TRAIN_FILE.exists() or not VALID_FILE.exists():
    raise FileNotFoundError('Dataset path is wrong or dataset is not attached to this Kaggle notebook.')

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('num_gpus:', NUM_GPUS)
for idx in range(NUM_GPUS):
    print(f'gpu {idx}:', torch.cuda.get_device_name(idx))

BASE_OVERRIDES = [
    f'data.train_file={TRAIN_FILE}',
    f'data.valid_file={VALID_FILE}',
]

RUN_CONFIGS = {}

def _override_string(items):
    return ' '.join(f'--set {item}' for item in items)

def train_config(run_name, config_path, overrides=None, force_single_gpu=False, overwrite=False):
    run_dir = OUTPUT_ROOT / run_name
    RUN_CONFIGS[run_name] = config_path
    if (run_dir / 'eval_results.json').exists() and not overwrite:
        print(f'SKIP {run_name}: metrics exist at {run_dir / "eval_results.json"}')
        return 0
    resume_override = []
    if run_dir.exists() and not overwrite:
        checkpoints = sorted(run_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1)
        if checkpoints:
            last_checkpoint = checkpoints[-1]
            print(f'RESUME {run_name} from {last_checkpoint}')
            resume_override = [f'training.resume_from_checkpoint={last_checkpoint}']
        else:
            print(f'CLEAR incomplete output without checkpoint: {run_dir}')
            shutil.rmtree(run_dir)
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={run_dir}'] + resume_override + (overrides or [])
    train_module_cmd = f'-m vn_summarization.train --config {config_path} {_override_string(final_overrides)}'
    if NUM_GPUS >= 2 and not force_single_gpu:
        cmd = (
            f'{sys.executable} -m accelerate.commands.launch '
            f'--multi_gpu --num_processes {NUM_GPUS} --mixed_precision fp16 '
            f'{train_module_cmd}'
        )
    else:
        cmd = f'{sys.executable} -u {train_module_cmd}'
    return run(cmd, cwd=repo)

def evaluate_run(eval_name, config_path, model_path, overrides=None, overwrite=False):
    eval_dir = OUTPUT_ROOT / eval_name
    if (eval_dir / 'validation_metrics.json').exists() and not overwrite:
        print(f'SKIP {eval_name}: metrics exist at {eval_dir / "validation_metrics.json"}')
        return 0
    eval_dir.mkdir(parents=True, exist_ok=True)
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={eval_dir}'] + (overrides or [])
    cmd = (
        f'{sys.executable} -u -m vn_summarization.evaluate '
        f'--config {config_path} '
        f'--model_path {model_path} '
        f'--predictions_path {eval_dir / "predictions_valid.jsonl"} '
        f'{_override_string(final_overrides)}'
    )
    return run(cmd, cwd=repo)

def summarize_results():
    return run(
        f'{sys.executable} -u -m vn_summarization.summarize_results --root {OUTPUT_ROOT}',
        cwd=repo,
        check=False,
    )


CMD: nvidia-smi
Mon Jun  8 18:45:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

## Tokenizer Check

ViT5 tokenizer phai pass truoc khi train. Neu cell nay fail thi khong chay train.


In [4]:
run(f'{sys.executable} -u scripts/check_tokenizer.py', cwd=repo)


CMD: /usr/bin/python3 -u scripts/check_tokenizer.py
T5Tokenizer
is_fast False
vocab_size 36096
pad/eos/unk 0 1 2
tokens 17


0

## Experiment Flags

Defaults are optimized for Kaggle T4x2 with spare VRAM: ViT5 full uses batch 4/GPU with grad accumulation 2, LoRA uses batch 8/GPU with accumulation 1. Effective batch stays about 16, so training dynamics stay close while wall time improves. If OOM happens, lower train/eval batch and raise accumulation.


In [5]:
# Time-budget friendly defaults for Kaggle T4x2.
# Uses more VRAM than the conservative config: full batch 4/GPU, LoRA batch 8/GPU.
# If OOM happens, reduce per_device_train_batch_size and increase gradient_accumulation_steps.
# Warm-start is optional and disabled by default to keep total runtime reasonable.
RUN_QUICK = False
RUN_VIT5_BASE = True
RUN_LORA = True
RUN_WARMSTART = False
RUN_BARTPHO = False
RUN_MT5 = False
OVERWRITE_RUNS = False

COMMON_T4_OVERRIDES = [
    'training.eval_steps=500',
    'training.save_steps=500',
    'training.logging_steps=100',
    'training.save_total_limit=2',
    'training.ddp_find_unused_parameters=false',
]

EXPERIMENTS = []

if RUN_QUICK:
    EXPERIMENTS.append({
        'name': 'vit5_base_quick20_t4x2',
        'config': 'configs/vit5_base.yaml',
        'overrides': COMMON_T4_OVERRIDES + [
            'training.max_steps=20',
            'training.eval_steps=10',
            'training.save_steps=10',
            'training.per_device_train_batch_size=4',
            'training.per_device_eval_batch_size=8',
            'training.gradient_accumulation_steps=2',
            'model.use_fast_tokenizer=false',
            'data.max_train_samples=128',
            'data.max_eval_samples=32',
        ],
    })

if RUN_VIT5_BASE:
    EXPERIMENTS.append({
        'name': 'vit5_base_ep3_t4x2',
        'config': 'configs/vit5_base.yaml',
        'overrides': COMMON_T4_OVERRIDES + [
            'training.num_train_epochs=3',
            'training.per_device_train_batch_size=4',
            'training.per_device_eval_batch_size=8',
            'training.gradient_accumulation_steps=2',
            'model.use_fast_tokenizer=false',
        ],
    })

if RUN_LORA:
    EXPERIMENTS.append({
        'name': 'vit5_base_lora_ep3_t4x2',
        'config': 'configs/vit5_base_lora.yaml',
        'overrides': COMMON_T4_OVERRIDES + [
            'training.num_train_epochs=3',
            'training.per_device_train_batch_size=8',
            'training.per_device_eval_batch_size=16',
            'training.gradient_accumulation_steps=1',
            'model.use_fast_tokenizer=false',
        ],
    })

if RUN_WARMSTART:
    EXPERIMENTS.append({
        'name': 'vit5_news_warmstart_ep2_t4x2',
        'config': 'configs/vit5_news_warmstart.yaml',
        'overrides': COMMON_T4_OVERRIDES + [
            'training.num_train_epochs=2',
            'training.eval_steps=400',
            'training.save_steps=400',
            'training.per_device_train_batch_size=4',
            'training.per_device_eval_batch_size=8',
            'training.gradient_accumulation_steps=2',
            'training.learning_rate=0.00001',
            'model.use_fast_tokenizer=false',
        ],
    })

if RUN_BARTPHO:
    EXPERIMENTS.append({
        'name': 'bartpho_syllable_ep2_t4x2',
        'config': 'configs/bartpho_syllable.yaml',
        'overrides': COMMON_T4_OVERRIDES + [
            'training.num_train_epochs=2',
            'training.per_device_train_batch_size=4',
            'training.per_device_eval_batch_size=8',
            'training.gradient_accumulation_steps=2',
        ],
    })

if RUN_MT5:
    EXPERIMENTS.append({
        'name': 'mt5_base_ep2_t4x2',
        'config': 'configs/mt5_base.yaml',
        'overrides': COMMON_T4_OVERRIDES + [
            'training.num_train_epochs=2',
            'training.per_device_train_batch_size=2',
            'training.per_device_eval_batch_size=4',
            'training.gradient_accumulation_steps=4',
        ],
    })

print(json.dumps(EXPERIMENTS, indent=2))


[
  {
    "name": "vit5_base_ep3_t4x2",
    "config": "configs/vit5_base.yaml",
    "overrides": [
      "training.eval_steps=500",
      "training.save_steps=500",
      "training.logging_steps=100",
      "training.save_total_limit=2",
      "training.ddp_find_unused_parameters=false",
      "training.num_train_epochs=3",
      "training.per_device_train_batch_size=4",
      "training.per_device_eval_batch_size=8",
      "training.gradient_accumulation_steps=2",
      "model.use_fast_tokenizer=false"
    ]
  },
  {
    "name": "vit5_base_lora_ep3_t4x2",
    "config": "configs/vit5_base_lora.yaml",
    "overrides": [
      "training.eval_steps=500",
      "training.save_steps=500",
      "training.logging_steps=100",
      "training.save_total_limit=2",
      "training.ddp_find_unused_parameters=false",
      "training.num_train_epochs=3",
      "training.per_device_train_batch_size=8",
      "training.per_device_eval_batch_size=16",
      "training.gradient_accumulation_steps=1",
   

## Run Selected Experiments

Cell nay tu skip run da co `eval_results.json`, nen rerun notebook khong train lai neu `OVERWRITE_RUNS=False`.


In [6]:
for exp in EXPERIMENTS:
    print('\n' + '=' * 100)
    print('RUN', exp['name'])
    train_config(
        run_name=exp['name'],
        config_path=exp['config'],
        overrides=exp.get('overrides', []),
        overwrite=OVERWRITE_RUNS,
    )
    summarize_results()



RUN vit5_base_ep3_t4x2
CMD: /usr/bin/python3 -m accelerate.commands.launch --multi_gpu --num_processes 2 --mixed_precision fp16 -m vn_summarization.train --config configs/vit5_base.yaml --set data.train_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/train-00000-of-00001.parquet --set data.valid_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/valid-00000-of-00001.parquet --set training.output_dir=/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2 --set training.eval_steps=500 --set training.save_steps=500 --set training.logging_steps=100 --set training.save_total_limit=2 --set training.ddp_find_unused_parameters=false --set training.num_train_epochs=3 --set training.per_device_train_batch_size=4 --set training.per_device_eval_batch_size=8 --set training.gradient_accumulation_steps=2 --set model.use_fast_tokenizer=false
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set t

## Summarize Main Results


In [7]:
summarize_results()

summary_csv = OUTPUT_ROOT / 'summary_results.csv'
summary_md = OUTPUT_ROOT / 'summary_results.md'
best_json = OUTPUT_ROOT / 'best_run.json'
print('summary_csv:', summary_csv)
print('summary_md:', summary_md)
print('best_json:', best_json)
if best_json.exists():
    print(best_json.read_text(encoding='utf-8'))


CMD: /usr/bin/python3 -u -m vn_summarization.summarize_results --root /kaggle/working/summarization_outputs
# Result Summary

| run | model | lora | rouge1 | rouge2 | rougeL | gen_len | loss |
| --- | --- | --- | --- | --- | --- | --- | --- |
| vit5_base_ep3_t4x2 | VietAI/vit5-base | False | 74.1679 | 47.0856 | 49.2356 | 125.1471 | 2.3506 |
| vit5_base_lora_ep3_t4x2 | VietAI/vit5-base | True | 72.9694 | 44.8377 | 47.3338 | 117.9753 | 2.5287 |

CSV: /kaggle/working/summarization_outputs/summary_results.csv
Best: /kaggle/working/summarization_outputs/best_run.json
summary_csv: /kaggle/working/summarization_outputs/summary_results.csv
summary_md: /kaggle/working/summarization_outputs/summary_results.md
best_json: /kaggle/working/summarization_outputs/best_run.json
{
  "run": "vit5_base_ep3_t4x2",
  "metric_file": "vit5_base_ep3_t4x2/eval_results.json",
  "model": "VietAI/vit5-base",
  "lora": false,
  "lr": 3e-05,
  "epochs": 3,
  "max_steps": -1,
  "source_len": 768,
  "target_len": 160,

## Decode Ablations On Best Model

Optional. Leave disabled while training main models. Enable after base/LoRA finish if Kaggle session still has time. These runs do not train, but full validation generation still costs time.


In [8]:
RUN_DECODE_ABLATIONS = False
OVERWRITE_EVALS = False

if RUN_DECODE_ABLATIONS:
    best = json.loads((OUTPUT_ROOT / 'best_run.json').read_text(encoding='utf-8'))
    best_run = best['run']
    best_model = OUTPUT_ROOT / best_run / 'best'
    best_config = RUN_CONFIGS.get(best_run)
    if best_config is None:
        if 'warmstart' in best_run:
            best_config = 'configs/vit5_news_warmstart.yaml'
        elif 'lora' in best_run:
            best_config = 'configs/vit5_base_lora.yaml'
        elif 'bartpho' in best_run:
            best_config = 'configs/bartpho_syllable.yaml'
        elif 'mt5' in best_run:
            best_config = 'configs/mt5_base.yaml'
        else:
            best_config = 'configs/vit5_base.yaml'
    print('best_run:', best_run)
    print('best_model:', best_model)
    print('best_config:', best_config)

    decode_abls = [
        ('decode_beam4_lp10_rp105', ['generation.num_beams=4', 'generation.length_penalty=1.0', 'generation.repetition_penalty=1.05']),
        ('decode_beam6_lp10_rp105', ['generation.num_beams=6', 'generation.length_penalty=1.0', 'generation.repetition_penalty=1.05']),
        ('decode_beam6_lp11_rp108', ['generation.num_beams=6', 'generation.length_penalty=1.1', 'generation.repetition_penalty=1.08']),
        ('decode_beam4_lp09_rp108', ['generation.num_beams=4', 'generation.length_penalty=0.9', 'generation.repetition_penalty=1.08']),
    ]
    for suffix, overrides in decode_abls:
        evaluate_run(
            eval_name=f'{best_run}_{suffix}',
            config_path=best_config,
            model_path=best_model,
            overrides=overrides,
            overwrite=OVERWRITE_EVALS,
        )
    summarize_results()


## Export Best Predictions

Ghi predictions JSONL cho error analysis trong bao cao.


In [9]:
best = json.loads((OUTPUT_ROOT / 'best_run.json').read_text(encoding='utf-8'))
best_run = best['run']
best_model = OUTPUT_ROOT / best_run / 'best'
if 'warmstart' in best_run:
    best_config = 'configs/vit5_news_warmstart.yaml'
elif 'lora' in best_run:
    best_config = 'configs/vit5_base_lora.yaml'
elif 'bartpho' in best_run:
    best_config = 'configs/bartpho_syllable.yaml'
elif 'mt5' in best_run:
    best_config = 'configs/mt5_base.yaml'
else:
    best_config = 'configs/vit5_base.yaml'

evaluate_run(
    eval_name=f'{best_run}_predictions_export',
    config_path=best_config,
    model_path=best_model,
    overrides=[],
    overwrite=False,
)
summarize_results()


CMD: /usr/bin/python3 -u -m vn_summarization.evaluate --config configs/vit5_base.yaml --model_path /kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/best --predictions_path /kaggle/working/summarization_outputs/vit5_base_ep3_t4x2_predictions_export/predictions_valid.jsonl --set data.train_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/train-00000-of-00001.parquet --set data.valid_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/valid-00000-of-00001.parquet --set training.output_dir=/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2_predictions_export
2026-06-09 00:58:10.915100: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780966690.938126     335 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E

0

## Save Version Outputs

Creates `summarization_results.zip`, `summarization_output_manifest.txt`, and `SAVE_VERSION_README.txt`. Kaggle Save Version will keep `/kaggle/working/summarization_outputs`, including best checkpoints.


In [10]:
zip_path = WORKING / 'summarization_results.zip'
manifest_path = WORKING / 'summarization_output_manifest.txt'
readme_path = WORKING / 'SAVE_VERSION_README.txt'

best_info = {}
if (OUTPUT_ROOT / 'best_run.json').exists():
    best_info = json.loads((OUTPUT_ROOT / 'best_run.json').read_text(encoding='utf-8'))

manifest_lines = [
    f'OUTPUT_ROOT={OUTPUT_ROOT}',
    f'BEST_RUN={best_info.get("run", "")}',
    f'BEST_CHECKPOINT={OUTPUT_ROOT / best_info.get("run", "") / "best" if best_info.get("run") else ""}',
    f'SUMMARY_CSV={OUTPUT_ROOT / "summary_results.csv"}',
    f'SUMMARY_MD={OUTPUT_ROOT / "summary_results.md"}',
    f'BEST_JSON={OUTPUT_ROOT / "best_run.json"}',
    '',
    'All output files:',
]
for file in sorted(OUTPUT_ROOT.rglob('*')):
    if file.is_file():
        manifest_lines.append(str(file))
manifest_path.write_text('\n'.join(manifest_lines), encoding='utf-8')

readme_path.write_text(
    '\n'.join([
        'Kaggle Save Version notes',
        '',
        'Keep /kaggle/working/summarization_outputs as notebook output.',
        'Best checkpoint folder is listed in summarization_output_manifest.txt.',
        'summarization_results.zip contains metrics, configs, summaries, and predictions only; checkpoints are not zipped to avoid huge archives.',
        'For report tables, use summarization_outputs/summary_results.csv or summary_results.md.',
    ]),
    encoding='utf-8',
)

if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
files += [manifest_path, readme_path]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP:', zip_path)
print('MANIFEST:', manifest_path)
print('README:', readme_path)
print('FILES:', len(files))
for file in sorted(files)[:80]:
    print(file)


ZIP: /kaggle/working/summarization_results.zip
MANIFEST: /kaggle/working/summarization_output_manifest.txt
README: /kaggle/working/SAVE_VERSION_README.txt
FILES: 52
/kaggle/working/SAVE_VERSION_README.txt
/kaggle/working/summarization_output_manifest.txt
/kaggle/working/summarization_outputs/best_run.json
/kaggle/working/summarization_outputs/summary_results.csv
/kaggle/working/summarization_outputs/summary_results.md
/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/all_results.json
/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/best/added_tokens.json
/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/best/config.json
/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/best/generation_config.json
/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/best/special_tokens_map.json
/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/best/tokenizer_config.json
/kaggle/working/summarization_outputs/vit5_base_ep3_t4x2/best/validation_metrics.json
/kaggle/w